# PathSnap · 05 histogram gradient boosting

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import f1_score, log_loss
from time import perf_counter

FEATURES = ['dist_m', 'dist_min_m', 'dist_rel_m', 'rank_dist', 'n_cands', 'proj_frac', 'arc_len_m', 'dist_end_m', 'head_diff_deg', 'head_cos', 'speed_mps', 'moving', 'dt_prev_s', 'dist_prev_pt_m', 'dist_next_pt_m', 'has_reverse', 'deg_from', 'deg_to']
KEYS = ["trip_id", "observation_index", "arc_id"]
ROOT = Path("pathsnap_real")
ROOT.mkdir(exist_ok=True)

def load_candidates(filename):
    if not Path(filename).exists():
        raise FileNotFoundError(f"Upload {filename} to the Colab Files panel.")
    frame = pd.read_csv(filename, dtype={"trip_id": str})
    missing = set(KEYS + FEATURES + ["label"]) - set(frame.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {sorted(missing)}")
    assert not frame[KEYS].isna().any().any(), f"Missing IDs in {filename}"
    assert not frame.duplicated(KEYS).any(), f"Duplicate candidate IDs in {filename}"
    assert frame["label"].isin([0, 1]).all(), f"Bad labels in {filename}"
    return frame

train = load_candidates("train_candidates.csv")
test_path = Path("student_test_candidates.csv")
if not test_path.exists():
    test_path = Path("student_test_candidates(1).csv")
test = load_candidates(test_path)

assert set(train.trip_id).isdisjoint(test.trip_id)
assert train.trip_id.nunique() >= 2
X, y = train[FEATURES], train.label
folds = list(GroupKFold(min(5, train.trip_id.nunique())).split(X, y, train.trip_id))
assert all(y.iloc[i].nunique() == 2 for i, _ in folds)

def evaluate(model):
    start = perf_counter()
    probabilities = model.predict_proba(test[FEATURES])
    seconds = perf_counter() - start
    scored = test[KEYS + ["label"]].copy()
    scored["p_match"] = probabilities[:, list(model.classes_).index(1)]
    top = scored.sort_values("p_match", ascending=False, kind="stable").drop_duplicates(KEYS[:2])
    return scored, {
        "student_top1_with_candidates": top.label.mean(),
        "student_candidate_macro_f1": f1_score(test.label, model.classes_[probabilities.argmax(axis=1)], average="macro"),
        "log_loss": log_loss(test.label, probabilities, labels=model.classes_),
        "inference_seconds": seconds,
    }


In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

NAME = "05_hist_gradient_boosting"
model = make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(random_state=42))
cv = cross_val_score(model, X, y, cv=folds, scoring="f1_macro", error_score="raise")
model.fit(X, y)
_, metrics = evaluate(model)
summary = pd.DataFrame([{"model": NAME, "cv_macro_f1": cv.mean(), **metrics}])
summary.to_csv(ROOT / f"{NAME}.csv", index=False)
joblib.dump(model, ROOT / f"{NAME}.joblib")
print(ROOT / f"{NAME}.csv", ROOT / f"{NAME}.joblib")
display(summary)


pathsnap_real/05_hist_gradient_boosting.csv pathsnap_real/05_hist_gradient_boosting.joblib


,model,cv_macro_f1,student_top1_with_candidates,student_candidate_macro_f1,log_loss,inference_seconds
0,05_hist_gradient_boosting,0.923231,0.872443,0.927751,0.055019,1.972549
